# Aerial Object Detection: Civilian Fine-Tuning

New reconstruction work, not historical training. This notebook fine-tunes a public torchvision detector on a group-separated HERIDAL export for offline civilian image experiments. It adds no control or navigation functionality.

The local pilot and Colab run are distinct experiments. This notebook has been syntax-checked locally, but has not been executed in your Colab account. Select a GPU runtime first. Free GPU availability is not guaranteed.

Build `outputs/aerial_detection_training.zip` locally with `.venv-level-b/bin/python scripts/build_training_bundle.py`, then upload that small archive below. Put your HERIDAL export folder (or a ZIP of it) in your own Google Drive. Datasets and weights are not included in the project ZIP.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import stat
import subprocess
import sys
import zipfile
from google.colab import files, drive


def safe_extract(archive_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination) or stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError('Unsafe archive path or symlink')
        archive.extractall(destination)

uploaded = files.upload()  # Upload aerial_detection_training.zip only.
project_zip = next(name for name in uploaded if name.endswith('.zip'))
safe_extract(project_zip, '/content/aerial_project')
PROJECT = Path('/content/aerial_project/aerial-object-detection')
assert (PROJECT / 'pyproject.toml').is_file()
print('Project ready:', PROJECT)

## Isolated Python and CUDA environment

The project pins Python-compatible Torch/torchvision versions. An isolated Python 3.11 environment avoids changing Colab's notebook kernel and works even if Colab's default Python changes. Downloads need internet access. The CUDA wheel source follows PyTorch's official previous-version instructions.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.8.22'], check=True)
subprocess.run(['uv', 'python', 'install', '3.11'], check=True)
subprocess.run(['uv', 'venv', '--python', '3.11', '--seed', '/content/aerial-env'], check=True)
PYTHON = Path('/content/aerial-env/bin/python')
subprocess.run(['uv', 'pip', 'install', '--python', str(PYTHON), 'torch==2.5.1', 'torchvision==0.20.1', '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)
subprocess.run(['uv', 'pip', 'install', '--python', str(PYTHON), '-r', 'requirements-training.txt'], cwd=PROJECT, check=True)
subprocess.run(['uv', 'pip', 'install', '--python', str(PYTHON), '-e', '.', '--no-deps'], cwd=PROJECT, check=True)


def run(*arguments):
    subprocess.run([str(PYTHON), *map(str, arguments)], cwd=PROJECT, check=True)

run('-m', 'pip', 'check')
run('-c', "import torch; assert torch.cuda.is_available(), 'Select a GPU runtime'; print(torch.__version__, torch.cuda.get_device_name(0))")

## Bring your own dataset

Mounting Drive requires your Google authorization. Adjust the two paths below. The archive is preferred for Drive I/O performance; an already uploaded export folder is also supported. Data is copied/extracted to the runtime disk; original Drive files are unchanged. Allow roughly 5 GB runtime storage for the export, plus the isolated environment and checkpoints.

In [ ]:
drive.mount('/content/drive')
DATA_ARCHIVE = Path('/content/drive/MyDrive/aerial-detection/heridal.zip')
DATA_FOLDER = Path('/content/drive/MyDrive/aerial-detection/Heridal.v2-yolo_non_412.coco')
DATASET = Path('/content/heridal')
if not DATASET.exists():
    if DATA_ARCHIVE.is_file():
        unpacked = Path('/content/heridal_unpacked')
        safe_extract(DATA_ARCHIVE, unpacked)
        candidates = list(unpacked.rglob('_annotations.coco.json'))
        if len(candidates) != 1:
            raise ValueError('Expected one COCO annotation file in the dataset archive')
        shutil.move(str(candidates[0].parent), str(DATASET))
    elif (DATA_FOLDER / '_annotations.coco.json').is_file():
        shutil.copytree(DATA_FOLDER, DATASET, ignore=shutil.ignore_patterns('.DS_Store'))
    else:
        raise FileNotFoundError('Upload the dataset ZIP or folder to Drive and adjust these paths')
assert (DATASET / 'train').is_dir()
run('-m', 'falcon_vision_reconstructed.training', 'prepare', '--dataset', DATASET, '--output', '/content/training_data')
print(Path('/content/training_data/audit.json').read_text())

## Training and automatic resume

The split reserves BRK/RAK for validation and VRD/ZRI for test; filename groups are only a scene proxy. Unverified unannotated images are excluded. This positive-only protocol cannot measure false positives on the full scene distribution.

The Colab config trains all configured trainable parts at 768-pixel input size, using all 535 audited training images. It monitors 8 fixed validation images, not the entire validation split, and stops after at most 20 epochs or validation patience. Checkpoints include optimizer state, progress and RNG states; latest/best/baseline files are written to Drive. A runtime interruption can lose steps since the last checkpoint. Changing resolution/mode/splits requires a new run folder.

In [ ]:
RUN_DIR = Path('/content/drive/MyDrive/aerial-detection/run1')
arguments = ['-m', 'falcon_vision_reconstructed.training', 'train', '--config', 'configs/training_colab.toml', '--image-root', DATASET / 'train', '--manifest', '/content/training_data/manifest.json', '--output-dir', RUN_DIR]
if (RUN_DIR / 'last.pt').is_file():
    arguments += ['--resume', RUN_DIR / 'last.pt']
run(*arguments)
print((RUN_DIR / 'summary.json').read_text())

## Optional final held-out evaluation

Leave the test set untouched while choosing settings. Enable this only after selecting a run. Full-frame grid inference on all held-out images can take substantially longer than validation on 8 images. Metrics are new experimental results, never historical project metrics. The best checkpoint can remain the initial baseline if fine-tuning did not improve validation AP.

In [ ]:
RUN_FULL_TEST = False
if RUN_FULL_TEST:
    run('-m', 'falcon_vision_reconstructed.training', 'evaluate', '--checkpoint', RUN_DIR / 'best.pt', '--manifest', '/content/training_data/manifest.json', '--image-root', DATASET / 'train', '--split', 'test', '--device', 'cuda', '--output', RUN_DIR / 'test_metrics.json')
    print((RUN_DIR / 'test_metrics.json').read_text())

## Use a saved checkpoint on a still image

This is offline visualization only. The example uses a published resized sample; it is not the full-resolution test benchmark. Verify predictions visually and do not treat a detected box as a recognized identity.

In [ ]:
run('-m', 'falcon_vision_reconstructed.training', 'infer', '--checkpoint', RUN_DIR / 'best.pt', '--input', 'assets/aerial/park.jpg', '--output', RUN_DIR / 'park_preview.jpg', '--device', 'cuda', '--threshold', '0.50')
from IPython.display import display, Image as DisplayImage
display(DisplayImage(filename=str(RUN_DIR / 'park_preview.jpg')))

## Sources

- Torchvision fine-tuning tutorial: https://docs.pytorch.org/tutorials/intermediate/torchvision_tutorial.html
- Official CUDA wheels: https://pytorch.org/get-started/previous-versions/
- Colab limits and checkpoint persistence considerations: https://research.google.com/colaboratory/faq.html
- Dataset provenance and licensing are in `assets/SAMPLE_SOURCE.md` in the project bundle.